In [ ]:
import geopandas as gpd
import matplotlib
import matplotlib.colors
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from mpl_toolkits.axes_grid1 import make_axes_locatable
import pandas as pd

from pathlib import Path
import os

In [ ]:
def plot_sectoral_GVA_admin_area(to_plot: gpd.GeoDataFrame, filename: str, quantile_drop: float=0.05):
    sectors = (
        ("1", "Primary", "YlGn"),
        ("2", "Secondary", "YlOrBr"),
        ("3", "Tertiary", "BuPu"),
    )

    f, axes = plt.subplots(3, 1, figsize=(8,11))
    cbar_formatter = ticker.ScalarFormatter()
    cbar_formatter.set_powerlimits((0, 0))
    for (sector, name, cmap_name), ax in zip(sectors, axes):
        cmap = plt.get_cmap(cmap_name)
        cmap.set_extremes(bad="pink", under="white", over="black")
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="5%", pad=0.05)
        to_plot.loc[:, [sector, "geometry"]].plot(
            sector,
            ax=ax,
            legend=True,
            cax=cax,
            cmap=cmap,
            legend_kwds={
                "extend": "both",
                "format": cbar_formatter,
            },
            vmin=to_plot.loc[:, [sector]].quantile(quantile_drop),
            vmax=to_plot.loc[:, [sector]].quantile(1 - quantile_drop),
        )
        ax.text(
            0.99,
            0.97,
            f"Total: {to_plot[sector].sum():1.1E} JMD d$^{{-1}}$",
            transform=ax.transAxes,
            horizontalalignment="right",
            verticalalignment="top",
            fontsize=8,
        )
        cax.set_ylabel(f"{name} GVA [JMD day$^{{-1}}$]")
        ax.grid(which="major", alpha=0.3)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['bottom'].set_visible(False)
        ax.spines['left'].set_visible(False)
        ax.xaxis.set_major_locator(ticker.MultipleLocator(0.5))
        ax.xaxis.set_minor_locator(ticker.MultipleLocator(0.1))
        ax.yaxis.set_major_locator(ticker.MultipleLocator(0.5))
        ax.yaxis.set_minor_locator(ticker.MultipleLocator(0.1))

    f.savefig(filename)
    plt.close(f)

In [ ]:
data_dir = Path("../../processed_data")

admin = {n: gpd.read_file(data_dir / "boundaries" / "admin_boundaries.gpkg", layer=f"admin{n}") for n in (1, 2, 3)}

jic = pd.read_csv("jic_mapping.csv", skiprows=3)

buildings = {}
buildings[2019] = gpd.read_file(data_dir / "buildings" / "buildings_assigned_economic_activity.gpkg")
buildings[2023] = gpd.read_parquet(data_dir / "buildings" / "buildings_assigned_economic_activity.geoparquet")

mines = {}
mines[2019] = gpd.read_file(data_dir / "mining_data" / "mining_gdp_2019.gpkg")
# Match schema of other inputs, i.e. GVA values under {sector_code}_GDP
# Use JIC 2016 classification for mining ('B')
# Convert 2023 figures to JMD / day, i.e. 2019's unit
df = gpd.read_file(data_dir / "mining_data" / "mining_gdp.gpkg")
df = df.rename(columns={"mining_gdp": "B_GDP"})
df.B_GDP = df.B_GDP * 1E6 / 365.25
mines[2023] = df

agri = {}
agri[2019] = gpd.read_file(data_dir / "agriculture_data" / "agriculture_gdp_2019.gpkg")
# Convert 2023 figures to JMD / day, i.e. 2019's unit
df = gpd.read_file(data_dir / "agriculture_data" / "agriculture_gdp.gpkg")
df.A_GDP = df.A_GDP * 1E6 / 365.25
agri[2023] = df

In [ ]:
gva = {}
b3 = {}
print("Sectoral GVA in B USD / year", end="\n\n")
for jic_year, gva_year in ((2005, 2019), (2016, 2023)):
    print(gva_year)

    gdp_cols = [c for c in buildings[gva_year].columns if (c.endswith("_GDP") and c != "total_GDP")]
    three, letter = jic.loc[:, ["Three sector", f"Sector ({jic_year})"]].values.T
    letter = [f"{s.strip()}_GDP" for s in letter]
    jic_three = {l: str(t) for l, t in zip(letter, three) if l in gdp_cols}

    cols = ["osm_id", "building_type", *jic_three.keys()]
    tmp = buildings[gva_year].loc[:, cols].copy().dropna(subset=["osm_id"])
    if tmp["osm_id"].duplicated().any():
        agg = {"building_type": "first", **{c: "sum" for c in jic_three.keys()}}
        tmp = tmp.groupby("osm_id", as_index=False).agg(agg)
    tmp = tmp.set_index("osm_id")

    gdp_df = tmp.loc[:, jic_three.keys()].copy()
    gdp_df.columns = pd.MultiIndex.from_arrays([[jic_three[c] for c in gdp_df.columns], gdp_df.columns])
    coarse = gdp_df.T.groupby(level=0).sum().T
    coarse["type"] = tmp["building_type"]
    b3[gva_year] = coarse.sort_index()

    # Primary sector GVA is mostly from agri and mining areas, not buildings
    primary = pd.concat(
        [
            agri[gva_year].rename(columns={"A_GDP": "1"}).loc[:, ["1", "geometry"]],
            # N.B. Mining sector code changes 2005 -> 2016 classification
            mines[gva_year].rename(columns={"C_GDP": "B_GDP"}).rename(columns={"B_GDP": "1"}).loc[:,["1", "geometry"]],
        ]
    )
    primary = primary.reset_index(drop=True)
    primary["2"] = 0
    primary["3"] = 0
    primary = primary.loc[:, ["1", "2", "3", "geometry"]]

    buildings_by_sector = gpd.GeoDataFrame(
        b3[gva_year].loc[:, ["1", "2", "3"]] \
            .join(buildings[gva_year].set_index("osm_id") \
            .loc[:, ["geometry"]])
    )
    gva[gva_year] = pd.concat(
        [
            # Agricultural and mining area GVA
            primary,
            # Secondary and Tertiary GVA is from buildings
            buildings_by_sector[~(buildings_by_sector.loc[:, "1"] > 0)],
        ]
    ).reset_index(names="gva_id")

    # B USD / year
    print(gva[gva_year].loc[:, ["1", "2", "3"]].sum() * 365 / 150 / 1e9, end="\n\n")


In [ ]:
for year in (2019, 2023):
    for admin_level, quantile_drop in ((1, 0), (2, 0.01), (3, 0.05)):

        cache_filename = f"{year}_sectoral_gva_admin_level_{admin_level}.gpq"
        if os.path.exists(cache_filename):
            to_plot = gpd.read_parquet(cache_filename)

        else:
            admin_geom = admin[admin_level].reset_index(names="admin_id")[["admin_id", "geometry"]]
            if admin_level < 3:
                # Intersect GVA representative points with admin areas, then sum GVA by sector for each admin area
                gva_point = gva[year].copy()
                gva_point.geometry = gva_point.geometry.representative_point()
                to_plot = gpd.GeoDataFrame(
                    gva_point.sjoin(admin_geom) \
                        .loc[:, ["1", "2", "3", "admin_id"]] \
                        .groupby("admin_id").sum() \
                        .join(admin_geom, on="admin_id")
                ).to_crs(4326)
                to_plot.to_parquet(cache_filename)
            else:
                # Some mining and agri areas may be larger than certain admin regions
                # In this case, split the GVA polygons on the admin polygons
                # This takes a few minutes
                splits = gpd.overlay(gva[year], admin_geom, how="intersection")

                # Now, apportion the GVA across the splits 
                splits["overlap_area"] = splits.geometry.area
                splits = splits.join(gva[year].set_index("gva_id").geometry.area.rename("gva_area"), on="gva_id")
                splits["weight"] = splits["overlap_area"] / splits["gva_area"]
                for col in ["1", "2", "3"]:
                    splits[col] = splits[col] * splits["weight"]
                to_plot = admin_geom.set_index("admin_id") \
                    .join(splits.groupby("admin_id")[["1", "2", "3"]].sum()) \
                    .to_crs(4326)
                to_plot.to_parquet(cache_filename)

        # Absolute value map, by sector
        plot_sectoral_GVA_admin_area(
            to_plot,
            f"{year}_sectoral_gva_admin_level_{admin_level}.png",
            quantile_drop=quantile_drop,
        )

In [ ]:
# Difference map, by sector
for admin_level, quantile_drop in ((1, 0), (2, 0.01), (3, 0.05)):
    div_cmap = "BrBG"
    sectors = (
        ("1", "Primary", div_cmap),
        ("2", "Secondary", div_cmap),
        ("3", "Tertiary", div_cmap),
    )

    a = gpd.read_parquet(f"2019_sectoral_gva_admin_level_{admin_level}.gpq")
    b = gpd.read_parquet(f"2023_sectoral_gva_admin_level_{admin_level}.gpq")
    gva_cols = ("1", "2", "3")
    to_plot = gpd.GeoDataFrame(
        (b.loc[:, gva_cols] - a.loc[:, gva_cols]) \
            .join(b.loc[:, ["geometry"]], on="admin_id")
    )

    f, axes = plt.subplots(3, 1, figsize=(8,11))
    for (sector, name, cmap_name), ax in zip(sectors, axes):
        cmap = plt.get_cmap(cmap_name)
        divider = make_axes_locatable(ax)
        cbar_formatter = ticker.ScalarFormatter()
        cbar_formatter.set_powerlimits((0, 0))
        cax = divider.append_axes("right", size="5%", pad=0.05)
        vmin = to_plot.loc[:, [sector]].quantile(quantile_drop).values.squeeze()
        vmax = to_plot.loc[:, [sector]].quantile(1 - quantile_drop).values.squeeze()
        norm = matplotlib.colors.CenteredNorm(0, max([vmax, -vmin]))
        to_plot.loc[:, [sector, "geometry"]].plot(
            sector,
            ax=ax,
            legend=True,
            cax=cax,
            cmap=cmap,
            norm=norm,
            legend_kwds={"extend": "both", "format": cbar_formatter},
        )
        cax.set_ylabel(f"$\Delta$ {name} GVA [JMD day$^{{-1}}$]")
        ax.grid(which="major", alpha=0.3)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['bottom'].set_visible(False)
        ax.spines['left'].set_visible(False)
        ax.xaxis.set_major_locator(ticker.MultipleLocator(0.5))
        ax.xaxis.set_minor_locator(ticker.MultipleLocator(0.1))
        ax.yaxis.set_major_locator(ticker.MultipleLocator(0.5))
        ax.yaxis.set_minor_locator(ticker.MultipleLocator(0.1))

    f.savefig(f"2019_to_2023_change_sectoral_gva_admin_level_{admin_level}.png")
    plt.close(f)